# Collective Foraging

> This short tutorial shows how to launch a multi-agent foraging training as done in [our last paper](https://arxiv.org/abs/2608.28046).

## High level launcher

The function `run_collective` works as a high-level API allowing to launch collective training with ease. It allows the control of most of the key parameters of the problem. Here is a typical usage example.

> **Important:** because of numba, the function needs to be compiled at first run. This takes around 1 minute. After that, the function has a 2 order of magnitude advantage w.r.t. a pure numpy run!

In [ ]:
from rl_opts.rl_framework.numba.agents import run_collective
import numpy as np

###### Environment Properties ######
L = 50 # Size of the environment
# Target properties
Nt = 100 # Number of targets
r = 0.5 # Radius of each target
tau = 3 # replenishment time
shared_depletion = False # If the target replenishment time is share among agents (True) or not (False). If False, each agent has its own target replenishment time.

###### Agents Properties ######
num_agents = 25 
tau_reward = 3 # Tag time after getting reward
agent_step = 1 # Length of the step of the agent
# Visual cone
visual_activated = True # If the vision is activate (turn off for testing the effect of vision)
visual_range = 4 # r_v in the paper, the distance at which the agent can see the target 
visual_angle=np.pi / 2 # Note that this is from the center, so from +45 to -45


###### Training parameters ######
max_counter = 50 # Maximum number of the counter for steps after turning. The state resets to 0 after reaching this.
gamma_damping = 0.00001 # PS gamma
eta_glow_damping = 0.1 # PS eta
# State space: [counter, any_agent_in_cone (0/1), rewarded_agent_in_cone (0/1)]
state_space = np.array([max_counter, 2, 2])

# Training length
# The results in the paper were obtained with time_ep = 5000 and episodes = 2000. We shorten here for educative purpose.
time_ep = 500
episodes = 20
# The function is built with parallel, multi-core running through numba. You tipycally want this to me a
# multiple of numba.get_num_threads() so every run of the loop has maximum number of threads used.
runs = 5

rewards, h_matrices = run_collective(episodes = episodes, time_ep = time_ep, 
                                    runs= runs,
                                    # Environment props
                                    Nt=Nt,
                                    L=L,
                                    r=r,
                                    tau=tau,
                                    tau_reward=tau_reward,
                                    num_agents=num_agents,
                                    agent_step=agent_step,
                                    visual_range=visual_range,
                                    visual_angle=visual_angle,
                                    shared_depletion=shared_depletion,
                                    visual_activated = visual_activated,
                                    # Agent props
                                    num_actions=2,
                                    state_space=state_space,
                                    gamma_damping=gamma_damping,
                                    eta_glow_damping=eta_glow_damping,
                                    )

The previous has two outputs: `rewards` and `h_matrics`. The first records the rewards for every run, agent and episode. For each episode, it return the **total** number of targets found in that episode.

In [ ]:
rewards.shape

(5, 25, 20)

`h_matrices` stores the Projective Simulation h-matrices. For every run and agent, we have a matrix of size (num_actions, state_space_size). The former will always be 2 in this case (turn and continue) while the later depends on the max_counter chosen.

In [ ]:
h_matrices.shape

(5, 25, 2, 200)

## Low level launching

Of course, the library allows for much more control in the definitions of agents and environments. For now, we will stick to the predefined training loop (see below), but the users are welcomed to look into the definition of that function to its adaption.

Here we do the same as above, but we define the agents and the environment separately, and run a single training with the properties defined.

In [ ]:
from rl_opts.rl_framework.numba.agents import Foragers_efficient, train_loop_collective
from rl_opts.rl_framework.numba.environments import CollectiveEnv

In [ ]:
# Agent definition
agents = Foragers_efficient( # This agents are prepared to be run in numba and contain adaptions on PS allowing for its most efficient update.
            num_agents, 2, state_space,
            gamma_damping, eta_glow_damping)

# Environment definition
env = CollectiveEnv(num_agents, Nt, L, r, tau, agent_step,
                    visual_range, visual_angle, shared_depletion, tau_reward)

# Launch training
rewards, h_matrices = train_loop_collective(episodes = episodes, 
                                            time_ep = time_ep, env = env, 
                                            agents = agents, 
                                            max_counter = state_space[0], 
                                            visual_activated = visual_activated)


Note that the above is a single run of the 5 parallel runs we launched with the `run_collective` function. 